# Dronomy localization framework -- demo

Runs the **plug-and-play framework** on the provided drone video: benchmarks
matchers through the shared data contract, reports field metrics, and shows a
map overlay. **Telemetry-free**: GPS is used only to *score*, never to localize.

Pipeline: `dataset -> Scenario -> (model x grid-search) -> FrameScore -> metrics -> best model`.

In [ ]:
import sys, pathlib
# Make `dronomy_loc` importable whether this runs from repo root or notebooks/.
for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (p / 'src' / 'dronomy_loc').exists():
        sys.path.insert(0, str(p / 'src')); break
import pandas as pd
from dronomy_loc.config import load_config
from dronomy_loc.datasets import get_dataset
from dronomy_loc.models import MODEL_NAMES, get_model
from dronomy_loc.framework.runner import run
from dronomy_loc.viz.figures import fig_model_comparison
from dronomy_loc.export import write_geojson, write_kml
cfg = load_config()
print('available models:', MODEL_NAMES)

## 1. Run the framework on the provided flight
We benchmark the dependency-free **SIFT** baseline on a few frames so the
notebook runs anywhere. Add `'loftr'` (needs torch+kornia) or `'roma'` (runs in
`docker/Dockerfile.matchanything`) to the model list to compare deep matchers.

In [ ]:
MODELS = ['sift']            # e.g. ['sift', 'loftr', 'roma'] with the right env
result = run(['video'], MODELS, cfg=cfg, select_metric='recall_5m', max_samples=6)
sc = result.scenarios[0]
print('scenario:', sc.scenario, '| terrain:', sc.terrain, '| best model:', sc.best_model)

## 2. Field metrics
Coverage (lock-rate), recall@5 m (coverage+accuracy KPI), median error, runtime,
and the SE(2)-aligned trajectory-shape ATE (the 'right shape and dimensions' metric).

In [ ]:
rows = [{'model': m,
         'coverage': round(f.lock_rate, 3),
         'recall@5m': round(f.recall_5m, 3),
         'median_err_m': None if f.median_err_m is None else round(f.median_err_m, 2),
         'mean_runtime_s': None if f.mean_runtime_s is None else round(f.mean_runtime_s, 2),
         'shape_ate_m': None if f.traj is None else round(f.traj.ate_aligned_m, 2)}
        for m, f in sc.per_model.items()]
pd.DataFrame(rows)

## 3. Model comparison figure

In [ ]:
fig_model_comparison(sc.per_model, 'framework_model_comparison.png', title=sc.scenario)
from IPython.display import Image
Image('framework_model_comparison.png')

## 4. Track export + map overlay
The best model's estimated track and the GT track, written to field formats
(GeoJSON / KML for Google Earth) and overlaid.

In [ ]:
best = sc.best_model or MODELS[0]
fs = sc.rows_by_model[best]
write_geojson(fs, 'framework_track.geojson', name=sc.scenario)
write_kml(fs, 'framework_track.kml', name=sc.scenario)
import matplotlib.pyplot as plt
est = [(r.est_lon, r.est_lat) for r in fs if r.est_lat is not None]
gt = [(r.gt_lon, r.gt_lat) for r in fs if r.gt_lon == r.gt_lon]
plt.figure(figsize=(6, 6))
if gt:  plt.plot([p[0] for p in gt], [p[1] for p in gt], 'g.-', label='ground truth')
if est: plt.plot([p[0] for p in est], [p[1] for p in est], 'r.-', label='estimate (%s)' % best)
plt.xlabel('longitude'); plt.ylabel('latitude'); plt.legend(); plt.title('Track overlay')
plt.gca().set_aspect('equal', 'datalim'); plt.show()

## 5. Generality notes
* **UAV-VisLoc** (`run(['uavvisloc'], MODELS, cfg=cfg)`) flies at ~466 m
  (~840 m footprint). The runner **scales the grid search per scene** from the
  altitude prior + FOV, so the same code localizes both the 50 m and 466 m flights.
* **RoMA** is the SOTA cross-modal matcher; its real weights run only in
  `docker/Dockerfile.matchanything`. Locally, SIFT/LoFTR are the runnable matchers.
* Everything here is **telemetry-free** -- GPS appears only in the scoring columns.